In [1]:
from opt_targeted_transfers import GapTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
# Gap targeted transfers
tt = GapTargetedTransfers(c_bar=2.15, n_regressors=5)

In [4]:
# Fit quantile regressors
tt.fit(train_dataset=train_dataset, validation_dataset=validation_dataset)

Fitting quantile regressor for quantile 0.05


100%|██████████| 300/300 [00:03<00:00, 97.42it/s, val loss=0.024]  


Fitting quantile regressor for quantile 0.27499999999999997


100%|██████████| 300/300 [00:03<00:00, 93.60it/s, val loss=0.107] 


Fitting quantile regressor for quantile 0.49999999999999994


100%|██████████| 300/300 [00:03<00:00, 88.78it/s, val loss=0.156]


Fitting quantile regressor for quantile 0.725


100%|██████████| 300/300 [00:03<00:00, 85.99it/s, val loss=0.169]


Fitting quantile regressor for quantile 0.95


100%|██████████| 300/300 [00:03<00:00, 90.82it/s, val loss=0.0827] 


In [5]:
# Get optimal policy by solving the optimization problem.
tt.set_budget(0.5)
tt.run_opt(test_covariate_dataset)
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.1739420959849197,
 'post_transfer_poverty_rate': 0.38018031717250844,
 'policy_cost_per_capita': 0.4999999999999979,
 'budget': 0.5,
 'policy_type': 'continuous_gap',
 'd': 2}

In [6]:
tt.set_budget(0.10)
tt.run_opt(test_covariate_dataset)
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.38054509464959346,
 'post_transfer_poverty_rate': 0.5455819686408198,
 'policy_cost_per_capita': 0.0999999999999991,
 'budget': 0.1,
 'policy_type': 'continuous_gap',
 'd': 2}

In [7]:
tt.compute_auc(test_covariate_dataset=test_covariate_dataset, test_dataset=test_dataset, metrics=["post_transfer_poverty_rate",
                                                              "post_transfer_poverty_gap"], budgets=[0.05, 0.1, 0.5, 1.0, 2.0])

{'post_transfer_poverty_rate': {'auc': 0.4463950241273305,
  'results': [0.5633239907546805,
   0.5455819686408198,
   0.38018031717250844,
   0.1541447242754854,
   0.04573259096007234]},
 'post_transfer_poverty_gap': {'auc': 0.20732351071450572,
  'results': [0.4115080300477348,
   0.38054509464959346,
   0.1739420959849197,
   0.04001520142934182,
   0.006255638803867166]}}